Wilcoxon signed-rank test across timepoints (PatientID is used to pair the samples)
-------------

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import wilcoxon # importing required statistical packages

df = pd.read_excel("PH_alphaDiversity_MASTER.xlsx", sheet_name="Sheet1")  # loading the Excel sheet into a dataframe

treatment_timepoints = ["Baseline", "Vanco", "IniMTT", "EndMTT", "FollUp"]  # defining treatment-group timepoints
placebo_timepoints = ["pl_Baseline", "pl_NoVanco", "pl_IniPart1"]  # defining placebo-group timepoints

Feature_cols = df.columns[17:]  # selecting feature columns after the metadata columns
print(Feature_cols)  # checking that only feature columns are included

print(df["TimePoints"].value_counts())  # checking for unexpected or misspelled timepoints

In [ ]:
def rank_biserial_effect_size(x, y):

    diffs = x - y  # computing paired differences
    diffs = diffs[~np.isclose(diffs, 0)]  # removing zero differences

    if len(diffs) == 0:
        return 0.0  # returning 0 if all paired values are identical

    abs_diffs = np.abs(diffs)  # taking absolute differences for ranking
    ranks = stats.rankdata(abs_diffs)  # assigning ranks to absolute differences

    signed_ranks = ranks * np.sign(diffs)  # restoring the sign to the ranked differences

    W_pos = np.sum(signed_ranks[signed_ranks > 0])  # summing positive signed ranks
    W_neg = -np.sum(signed_ranks[signed_ranks < 0])  # summing negative signed ranks

    r_b = (W_pos - W_neg) / (W_pos + W_neg)  # calculating rank-biserial correlation effect size
    return r_b

In [ ]:
def perform_wilcoxon(data, timepoints):

    comparisons = [
        (timepoints[i], timepoints[j])
        for i in range(len(timepoints))
        for j in range(i + 1, len(timepoints))
    ]  # generating all pairwise timepoint comparisons

    results = [] 

    for feature in Feature_cols:  
        for t1, t2 in comparisons:  

            df1 = data[data["TimePoints"] == t1][["PatientID", feature]]  # data for first timepoint
            df2 = data[data["TimePoints"] == t2][["PatientID", feature]]  # data for second timepoint

            merged = pd.merge(df1, df2, on="PatientID", suffixes=("_t1", "_t2"))  # pairing samples by PatientID
            merged = merged.dropna(subset=[f"{feature}_t1", f"{feature}_t2"])  

            total_n = len(merged)  

            if total_n > 0:

                values_t1 = merged[f"{feature}_t1"].values  
                values_t2 = merged[f"{feature}_t2"].values  

                effective_n = np.sum(~np.isclose(values_t1, values_t2))  # counting non-identical paired observations

                if effective_n == 0:
                    results.append([
                        feature, t1, t2, np.nan, np.nan, "Identical Values",
                        0.0, 0.0, total_n, effective_n
                    ])  # skipping cases where all paired values are identical
                    continue

                try:

                    stat, p_value = wilcoxon(
                        values_t1,
                        values_t2,
                        alternative="two-sided",  # Can be either "less" or "greater" based on the hypothesis
                        zero_method="wilcox",  # excluding zero differences before ranking
                        method="exact" # computing the exact p-value from the Wilcoxon distribution rather than using the normal approximation for smaller sample sizes )
                    )

                    rb = rank_biserial_effect_size(values_t1, values_t2) 

                    results.append([
                        feature, t1, t2, stat, p_value, "Tested",
                        rb, np.abs(rb), total_n, effective_n
                    ])  

                except ValueError as e:

                    print(f"Error in feature {feature}, {t1} vs {t2}: {e}")  

                    results.append([
                        feature, t1, t2, np.nan, np.nan, f"Error: {str(e)}",
                        np.nan, np.nan, total_n, effective_n
                    ]) 

    return pd.DataFrame(results, columns=[
        "feature", "TimePoint1", "TimePoint2", "Wilcoxon_Stat", "P_Value",
        "Status", "Rank_Biserial_r", "Rank_Biserial_r_Abs", "Total_N", "Effective_N"
    ])  

In [ ]:
treatment_results = perform_wilcoxon(df, treatment_timepoints)  # running Wilcoxon signed-rank test for the treatment group
placebo_results = perform_wilcoxon(df, placebo_timepoints)  # running Wilcoxon signed-rank test for the placebo group

all_results = pd.concat([treatment_results, placebo_results]) 

output_file = "wilcoxon_pairwise_less_at_tp1.xlsx" 
all_results.to_excel(output_file, index=False) 

print("Total valid Wilcoxon tests:", all_results["P_Value"].notna().sum()) 

print(
    "Total identical-value cases (skipped):",
    all_results["Status"].value_counts().get("Identical Values", 0)
)  

In the Wilcoxon signed-rank test, the zero_method="wilcox" option removes pairs where the difference between timepoints is zero, meaning patients who show no change are excluded from the ranking and test statistic. This is beneficial because the test is designed to assess whether there is a consistent direction of change among observations that actually vary; including zeros would dilute this signal without contributing meaningful information. When both the test and the effect size are computed after removing zero-difference pairs, the results should be interpreted as applying specifically to the subgroup of patients who exhibited a change. In this context, a significant p-value and a large rank-biserial correlation indicate a strong and consistent directional effect among those changing individuals, but they do not reflect the magnitude or prevalence of change in the full population.